# Gießen 6-Day Weather: Multi-Source Comparison & Trust-Weighted Median

1. Fetches forecasts for Gießen from several **free, no-API-key** weather APIs.
2. Normalizes them into one common table (per day: tmin, tmax, precip probability, precip sum, wind).
3. Assigns each source a **trust score** (a heuristic, see note below — there's no ground truth for a future forecast, so this is reputation/methodology-based, not measured).
4. Computes a trust-weighted median across sources.
5. Produces a ranked list of sources by trust score.


In [ ]:
%pip install requests
%pip install pandas

In [1]:
import requests
import pandas as pd
from datetime import datetime, timedelta

LAT, LON = 50.5841, 8.6782  # Gießen, Germany
TODAY = datetime.utcnow().date()
DAYS = [TODAY + timedelta(days=i) for i in range(6)]
print("Forecast window:", DAYS)

Forecast window: [datetime.date(2026, 6, 28), datetime.date(2026, 6, 29), datetime.date(2026, 6, 30), datetime.date(2026, 7, 1), datetime.date(2026, 7, 2), datetime.date(2026, 7, 3)]


/var/folders/98/jy1ytfqs1lvgg8y3k1rw_2t00000gn/T/ipykernel_87042/418085860.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  TODAY = datetime.utcnow().date()


## 1. Fetch functions (one per source)

Each function returns a list of dicts with keys: `date, tmin, tmax, precip_sum, precip_prob, wind_max, source`.

In [2]:
def fetch_open_meteo(lat, lon):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum,precipitation_probability_max,windspeed_10m_max",
        "timezone": "Europe/Berlin",
        "forecast_days": 6,
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    d = r.json()["daily"]
    out = []
    for i, date in enumerate(d["time"]):
        out.append({
            "date": date,
            "tmin": d["temperature_2m_min"][i],
            "tmax": d["temperature_2m_max"][i],
            "precip_sum": d["precipitation_sum"][i],
            "precip_prob": d.get("precipitation_probability_max", [None]*3)[i],
            "wind_max": d["windspeed_10m_max"][i],
            "source": "Open-Meteo",
        })
    return out


def fetch_brightsky(lat, lon):
    url = "https://api.brightsky.dev/weather"
    rows = []
    for day in DAYS:
        params = {"lat": lat, "lon": lon, "date": day.isoformat()}
        r = requests.get(url, params=params, timeout=15)
        r.raise_for_status()
        hourly = r.json().get("weather", [])
        if not hourly:
            continue
        temps = [h["temperature"] for h in hourly if h.get("temperature") is not None]
        precs = [h["precipitation"] for h in hourly if h.get("precipitation") is not None]
        winds = [h["wind_speed"] for h in hourly if h.get("wind_speed") is not None]
        probs = [h["precipitation_probability"] for h in hourly if h.get("precipitation_probability") is not None]
        if not temps:
            continue
        rows.append({
            "date": day.isoformat(),
            "tmin": min(temps),
            "tmax": max(temps),
            "precip_sum": sum(precs) if precs else None,
            "precip_prob": max(probs) if probs else None,
            "wind_max": max(winds) if winds else None,
            "source": "Bright Sky (DWD)",
        })
    return rows


def fetch_metno(lat, lon):
    url = "https://api.met.no/weatherapi/locationforecast/2.0/compact"
    headers = {"User-Agent": "giessen-weather-comparison/1.0 (personal research use)"}
    params = {"lat": lat, "lon": lon}
    r = requests.get(url, params=params, headers=headers, timeout=15)
    r.raise_for_status()
    series = r.json()["properties"]["timeseries"]

    by_day = {}
    for entry in series:
        ts = entry["time"]
        date = ts[:10]
        if date not in [d.isoformat() for d in DAYS]:
            continue
        details = entry["data"]["instant"]["details"]
        temp = details.get("air_temperature")
        wind = details.get("wind_speed")
        precip = None
        if "next_1_hours" in entry["data"]:
            precip = entry["data"]["next_1_hours"]["details"].get("precipitation_amount")
        by_day.setdefault(date, {"temps": [], "winds": [], "precs": []})
        if temp is not None:
            by_day[date]["temps"].append(temp)
        if wind is not None:
            by_day[date]["winds"].append(wind)
        if precip is not None:
            by_day[date]["precs"].append(precip)

    out = []
    for date, v in by_day.items():
        if not v["temps"]:
            continue
        out.append({
            "date": date,
            "tmin": min(v["temps"]),
            "tmax": max(v["temps"]),
            "precip_sum": sum(v["precs"]) if v["precs"] else None,
            "precip_prob": None,
            "wind_max": max(v["winds"]) if v["winds"] else None,
            "source": "MET Norway",
        })
    return out


def fetch_7timer(lat, lon):
    url = "http://www.7timer.info/bin/api.pl"
    params = {"lon": lon, "lat": lat, "product": "civillight", "output": "json"}
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    data = r.json().get("dataseries", [])
    out = []
    for i, entry in enumerate(data[:6]):
        date = (TODAY + timedelta(days=i)).isoformat()
        out.append({
            "date": date,
            "tmin": entry.get("temp2m", {}).get("min"),
            "tmax": entry.get("temp2m", {}).get("max"),
            "precip_sum": None,
            "precip_prob": None,
            "wind_max": None,
            "source": "7Timer",
        })
    return out

## 2. Fetch everything and combine into one table

In [3]:
fetchers = {
    "Open-Meteo": fetch_open_meteo,
    "Bright Sky (DWD)": fetch_brightsky,
    "MET Norway": fetch_metno,
    "7Timer": fetch_7timer,
}

all_rows = []
errors = {}
for name, fn in fetchers.items():
    try:
        rows = fn(LAT, LON)
        all_rows.extend(rows)
        print(f"OK  {name}: {len(rows)} day(s)")
    except Exception as e:
        errors[name] = str(e)
        print(f"FAIL {name}: {e}")

df = pd.DataFrame(all_rows)
df

OK  Open-Meteo: 6 day(s)
OK  Bright Sky (DWD): 6 day(s)
OK  MET Norway: 6 day(s)
OK  7Timer: 6 day(s)


,date,tmin,tmax,precip_sum,precip_prob,wind_max,source
0,2026-06-28,22.7,35.6,0.0,5.0,11.0,Open-Meteo
1,2026-06-29,19.9,26.8,18.4,68.0,11.6,Open-Meteo
2,2026-06-30,17.0,29.8,0.0,23.0,9.0,Open-Meteo
3,2026-07-01,16.9,25.3,1.7,65.0,13.2,Open-Meteo
4,2026-07-02,11.4,26.5,0.0,6.0,14.1,Open-Meteo
5,2026-07-03,14.6,23.0,2.1,15.0,18.8,Open-Meteo
6,2026-06-28,22.1,35.2,0.0,7.0,23.0,Bright Sky (DWD)
7,2026-06-29,17.7,26.2,0.0,29.0,11.1,Bright Sky (DWD)
8,2026-06-30,16.3,29.7,2.0,30.0,11.1,Bright Sky (DWD)
9,2026-07-01,15.8,25.5,0.0,29.0,18.5,Bright Sky (DWD)


## 3. Trust scores
- **Bright Sky (DWD)** — directly from Germany's national met service, the agency with the most localized model + station data for this exact region. Highest trust.
- **MET Norway** — another national agency with solid open-data global coverage, but not Germany-specific.
- **Open-Meteo** — aggregates outputs from several big physics models (ICON, GFS, etc.) rather than running its own; very good in practice but is a step removed from a single agency's QA process.
- **7Timer** — small hobby-run service, least transparent methodology, included mainly for breadth/comparison.

In [4]:
TRUST_SCORES = {
    "Bright Sky (DWD)": 0.95,
    "MET Norway": 0.85,
    "Open-Meteo": 0.80,
    "7Timer": 0.50,
}

df["trust"] = df["source"].map(TRUST_SCORES)

ranklist = (
    pd.DataFrame({"source": TRUST_SCORES.keys(), "trust_score": TRUST_SCORES.values()})
    .sort_values("trust_score", ascending=False)
    .reset_index(drop=True)
)
print("Source rank list by trust score:")
ranklist

Source rank list by trust score:


,source,trust_score
0,Bright Sky (DWD),0.95
1,MET Norway,0.85
2,Open-Meteo,0.80
3,7Timer,0.50


## 4. Plain (unweighted) median per day & variable

In [5]:
value_cols = ["tmin", "tmax", "precip_sum", "precip_prob", "wind_max"]
median_df = df.groupby("date")[value_cols].median().reset_index()
median_df.insert(1, "method", "plain_median")
median_df

,date,method,tmin,tmax,precip_sum,precip_prob,wind_max
0,2026-06-28,plain_median,23.80,34.60,0.00,6.0,11.0
1,2026-06-29,plain_median,18.85,26.50,3.50,48.5,11.1
2,2026-06-30,plain_median,16.65,29.40,0.30,26.5,9.0
3,2026-07-01,plain_median,16.35,25.45,0.30,47.0,13.2
4,2026-07-02,plain_median,12.75,26.35,0.00,8.0,14.1
5,2026-07-03,plain_median,14.45,23.05,1.05,15.5,16.7


## 5. Trust-weighted median

A true weighted *median* (not mean) — sorts values by magnitude, picks the value at which cumulative trust weight crosses 50%. More robust to one source being an outlier than a weighted mean.

In [6]:
def weighted_median(values, weights):
    pairs = sorted(zip(values, weights), key=lambda x: x[0])
    cum = 0
    total = sum(w for _, w in pairs)
    if total == 0:
        return None
    for val, w in pairs:
        cum += w
        if cum >= total / 2:
            return val
    return pairs[-1][0]

weighted_rows = []
for date, group in df.groupby("date"):
    row = {"date": date, "method": "trust_weighted_median"}
    for col in value_cols:
        sub = group.dropna(subset=[col])
        if sub.empty:
            row[col] = None
        else:
            row[col] = weighted_median(sub[col].tolist(), sub["trust"].tolist())
    weighted_rows.append(row)

weighted_df = pd.DataFrame(weighted_rows)
weighted_df

,date,method,tmin,tmax,precip_sum,precip_prob,wind_max
0,2026-06-28,trust_weighted_median,22.7,35.2,0.0,7.0,11.0
1,2026-06-29,trust_weighted_median,19.7,26.8,3.5,29.0,11.1
2,2026-06-30,trust_weighted_median,17.0,29.7,0.3,30.0,9.0
3,2026-07-01,trust_weighted_median,16.9,25.4,0.3,29.0,13.2
4,2026-07-02,trust_weighted_median,14.1,26.2,0.0,10.0,14.1
5,2026-07-03,trust_weighted_median,14.6,23.1,0.0,16.0,16.7


## 6. Side-by-side: every source vs. the two medians

In [7]:
comparison = pd.concat([
    df[["date", "source"] + value_cols].rename(columns={"source": "method"}),
    median_df,
    weighted_df,
], ignore_index=True)

comparison = comparison.sort_values(["date", "method"]).reset_index(drop=True)
comparison

,date,method,tmin,tmax,precip_sum,precip_prob,wind_max
0,2026-06-28,7Timer,25.00,34.00,NaN,NaN,NaN
1,2026-06-28,Bright Sky (DWD),22.10,35.20,0.00,7.0,23.0
2,2026-06-28,MET Norway,24.90,27.00,0.00,NaN,3.4
3,2026-06-28,Open-Meteo,22.70,35.60,0.00,5.0,11.0
4,2026-06-28,plain_median,23.80,34.60,0.00,6.0,11.0
5,2026-06-28,trust_weighted_median,22.70,35.20,0.00,7.0,11.0
6,2026-06-29,7Timer,18.00,24.00,NaN,NaN,NaN
7,2026-06-29,Bright Sky (DWD),17.70,26.20,0.00,29.0,11.1
8,2026-06-29,MET Norway,19.70,27.30,3.50,NaN,5.4
9,2026-06-29,Open-Meteo,19.90,26.80,18.40,68.0,11.6


## 7. Persist this run to a growing JSON store

appends the full `comparison` table under a new run ID

In [8]:
import json
import os
from datetime import datetime

STORE_PATH = "giessen_weather_history.json"

GERMAN_WEEKDAYS = {
    0: "Mo.", 1: "Di.", 2: "Mi.", 3: "Do.",
    4: "Fr.", 5: "Sa.", 6: "So.",
}

def make_run_id(dt=None):
    dt = dt or datetime.now()
    weekday = GERMAN_WEEKDAYS[dt.weekday()]
    return f"{weekday} {dt.day}. {dt.strftime('%B')} {dt.year}:{dt.strftime('%H/%M')}"


def load_store(path=STORE_PATH):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {}


def save_run(comparison_df, path=STORE_PATH):
    store = load_store(path)
    run_id = make_run_id()

    records = comparison_df.reset_index(drop=True).reset_index().rename(columns={"index": "row_id"})
    records = records.where(pd.notnull(records), None)

    store[run_id] = records.to_dict(orient="records")

    with open(path, "w", encoding="utf-8") as f:
        json.dump(store, f, indent=2, ensure_ascii=False)

    print(f"Saved run '{run_id}' with {len(records)} rows to {path}")
    print(f"Store now has {len(store)} run(s) total.")
    return run_id


current_run_id = save_run(comparison)
current_run_id

Saved run 'So. 28. June 2026:23/02' with 36 rows to giessen_weather_history.json
Store now has 1 run(s) total.


'So. 28. June 2026:23/02'

### lurkygirling at the growing history

In [9]:
def history_as_dataframe(path=STORE_PATH):
    store = load_store(path)
    frames = []
    for run_id, rows in store.items():
        sub = pd.DataFrame(rows)
        sub.insert(0, "run_id", run_id)
        frames.append(sub)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

history_df = history_as_dataframe()
history_df

,run_id,row_id,date,method,tmin,tmax,precip_sum,precip_prob,wind_max
0,So. 28. June 2026:23/02,0,2026-06-28,7Timer,25.00,34.00,NaN,NaN,NaN
1,So. 28. June 2026:23/02,1,2026-06-28,Bright Sky (DWD),22.10,35.20,0.00,7.0,23.0
2,So. 28. June 2026:23/02,2,2026-06-28,MET Norway,24.90,27.00,0.00,NaN,3.4
3,So. 28. June 2026:23/02,3,2026-06-28,Open-Meteo,22.70,35.60,0.00,5.0,11.0
4,So. 28. June 2026:23/02,4,2026-06-28,plain_median,23.80,34.60,0.00,6.0,11.0
5,So. 28. June 2026:23/02,5,2026-06-28,trust_weighted_median,22.70,35.20,0.00,7.0,11.0
6,So. 28. June 2026:23/02,6,2026-06-29,7Timer,18.00,24.00,NaN,NaN,NaN
7,So. 28. June 2026:23/02,7,2026-06-29,Bright Sky (DWD),17.70,26.20,0.00,29.0,11.1
8,So. 28. June 2026:23/02,8,2026-06-29,MET Norway,19.70,27.30,3.50,NaN,5.4
9,So. 28. June 2026:23/02,9,2026-06-29,Open-Meteo,19.90,26.80,18.40,68.0,11.6


# WEATHER REPORTO!!

In [10]:
WEEKDAY_DE = ["Montag", "Dienstag", "Mittwoch", "Donnerstag", "Freitag", "Samstag", "Sonntag"]

def rain_icon(precip_sum, precip_prob):
    if precip_sum is None and precip_prob is None:
        return "❓"
    p = precip_prob if precip_prob is not None else (40 if (precip_sum or 0) > 0 else 0)
    s = precip_sum or 0
    if s >= 5 or p >= 70:
        return "🌧️"
    if s > 0 or p >= 30:
        return "🌦️"
    return "☀️"

def wind_note(wind_max):
    if wind_max is None:
        return ""
    if wind_max >= 40:
        return " 💨 windy"
    if wind_max >= 20:
        return " 🍃 breezy"
    return ""

print("=" * 46)
print(f"  GIEẞEN — 6-DAY OUTLOOK (trust-weighted)")
print(f"  run: {current_run_id}")
print("=" * 46)

for _, row in weighted_df.sort_values("date").iterrows():
    date_obj = datetime.strptime(row["date"], "%Y-%m-%d")
    weekday = WEEKDAY_DE[date_obj.weekday()]
    icon = rain_icon(row["precip_sum"], row["precip_prob"])
    wind = wind_note(row["wind_max"])

    tmin = f"{row['tmin']:.0f}°" if pd.notnull(row["tmin"]) else "–"
    tmax = f"{row['tmax']:.0f}°" if pd.notnull(row["tmax"]) else "–"
    prob = f"{row['precip_prob']:.0f}%" if pd.notnull(row["precip_prob"]) else "–"
    rain_mm = f"{row['precip_sum']:.1f}mm" if pd.notnull(row["precip_sum"]) else "–"

    print(f"\n {icon}  {weekday}, {date_obj.strftime('%d. %B')}{wind}")
    print(f"    {tmin} – {tmax}   |   rain chance {prob}   |   {rain_mm}")

print("\n" + "=" * 46)
print(" Methodology: trust-weighted median across")
print(" Bright Sky (DWD), MET Norway, Open-Meteo, 7Timer")
print("=" * 46)

  GIEẞEN — 6-DAY OUTLOOK (trust-weighted)
  run: So. 28. June 2026:23/02

 ☀️  Sonntag, 28. June
    23° – 35°   |   rain chance 7%   |   0.0mm

 🌦️  Montag, 29. June
    20° – 27°   |   rain chance 29%   |   3.5mm

 🌦️  Dienstag, 30. June
    17° – 30°   |   rain chance 30%   |   0.3mm

 🌦️  Mittwoch, 01. July
    17° – 25°   |   rain chance 29%   |   0.3mm

 ☀️  Donnerstag, 02. July
    14° – 26°   |   rain chance 10%   |   0.0mm

 ☀️  Freitag, 03. July
    15° – 23°   |   rain chance 16%   |   0.0mm

 Methodology: trust-weighted median across
 Bright Sky (DWD), MET Norway, Open-Meteo, 7Timer


## 8. Measured trust scores from your accumulated history

run after logging everyday for like 2 weeks
1. Fetches what **actually happened** that day from Bright Sky's historical/observed station data (DWD ground station — the closest thing to ground truth we have access to).
2. Compares it to what each source predicted, **for that lead time** (1 day ahead, 2 days ahead, ... 6 days ahead).
3. Aggregates into a mean absolute error (MAE) per source, optionally broken down by lead time.
4. Converts MAE into a trust score you can feed back into `TRUST_SCORES` above.

In [ ]:
def fetch_actual(date_str, lat=LAT, lon=LON):
    url = "https://api.brightsky.dev/weather"
    params = {"lat": lat, "lon": lon, "date": date_str}
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    hourly = r.json().get("weather", [])
    temps = [h["temperature"] for h in hourly if h.get("temperature") is not None]
    precs = [h["precipitation"] for h in hourly if h.get("precipitation") is not None]
    if not temps:
        return None
    return {
        "date": date_str,
        "tmin": min(temps),
        "tmax": max(temps),
        "precip_sum": sum(precs) if precs else None,
    }


def run_id_to_datetime(run_id):
    _, rest = run_id.split(" ", 1)
    date_part, time_part = rest.split(":")
    day, month, year = date_part.replace(".", "").split()
    hour, minute = time_part.split("/")
    return datetime.strptime(f"{day} {month} {year} {hour}:{minute}", "%d %B %Y %H:%M")


def build_error_table(path=STORE_PATH):
    store = load_store(path)
    today = datetime.now().date()
    actual_cache = {}
    rows = []

    for run_id, records in store.items():
        run_dt = run_id_to_datetime(run_id)
        for rec in records:
            target_date = datetime.strptime(rec["date"], "%Y-%m-%d").date()
            if target_date >= today:
                continue
            if rec["method"] in ("plain_median", "trust_weighted_median"):
                continue
            if rec.get("tmax") is None:
                continue

            if target_date not in actual_cache:
                actual_cache[target_date] = fetch_actual(target_date.isoformat())
            actual = actual_cache[target_date]
            if actual is None:
                continue

            lead_days = (target_date - run_dt.date()).days
            rows.append({
                "source": rec["method"],
                "target_date": rec["date"],
                "lead_days": lead_days,
                "pred_tmax": rec["tmax"],
                "actual_tmax": actual["tmax"],
                "abs_err_tmax": abs(rec["tmax"] - actual["tmax"]),
                "pred_tmin": rec["tmin"],
                "actual_tmin": actual["tmin"],
                "abs_err_tmin": abs(rec["tmin"] - actual["tmin"]) if rec.get("tmin") is not None else None,
            })

    return pd.DataFrame(rows)


error_df = build_error_table()
error_df

### Aggregate into MAE per source, and per source+lead-time

In [ ]:
if error_df.empty:
    print("No scoreable history yet — once a target date from a past run has actually occurred, it'll show up here.")
else:
    mae_by_source = (
        error_df.groupby("source")["abs_err_tmax"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "mae_tmax", "count": "n_samples"})
        .sort_values("mae_tmax")
    )
    print("Overall MAE per source (°C, tmax):")
    display(mae_by_source)

    mae_by_source_lead = (
        error_df.groupby(["source", "lead_days"])["abs_err_tmax"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "mae_tmax", "count": "n_samples"})
    )
    print("\nMAE per source, broken down by lead time (days ahead):")
    display(mae_by_source_lead)

### Convert MAE into a measured trust score

Same shape as the original heuristic (higher = more trusted), but now derived from actual error: `trust = 1 / (1 + mae)`. A source with 0°C average error gets trust 1.0; one with 4°C average error gets trust 0.25

In [ ]:
if not error_df.empty:
    measured_trust = (1 / (1 + mae_by_source["mae_tmax"])).round(3).to_dict()
    print("Measured trust scores (based on", mae_by_source["n_samples"].to_dict(), "samples):")
    print(measured_trust)
    print("\nCopy-pasteable:")
    print("TRUST_SCORES = " + json.dumps(measured_trust, indent=4))